# 00 - PINN 预备知识：PyTorch、神经网络与自动微分

PINN 的核心工具很简单：

- 用神经网络表示未知函数，例如 `u_theta(x)`。
- 用自动微分计算导数，例如 `du/dx`、`d2u/dx2`。
- 把微分方程残差、边界条件、初值条件写成损失函数。
- 用梯度下降训练网络。

这个 notebook 先不解微分方程，只把这些工具跑通。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. 自动微分：计算函数导数

假设 `y = x^3 + 2x`，真实导数是 `dy/dx = 3x^2 + 2`。

In [ ]:
x = torch.linspace(-2, 2, 5, device=device).reshape(-1, 1)
x.requires_grad_(True)

y = x**3 + 2 * x
dy_dx = torch.autograd.grad(
    y,
    x,
    grad_outputs=torch.ones_like(y),
    create_graph=True
)[0]

print("x =", x.detach().cpu().flatten())
print("autograd dy/dx =", dy_dx.detach().cpu().flatten())
print("exact dy/dx =", (3 * x**2 + 2).detach().cpu().flatten())

## 2. 定义一个小型 MLP

PINN 通常用多层感知机 MLP 表示未知函数。这里先拟合一个普通函数 `sin(x)`。

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=1, out_dim=1, hidden_dim=32, depth=3):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
x_train = torch.linspace(-torch.pi, torch.pi, 80, device=device).reshape(-1, 1)
y_train = torch.sin(x_train)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_history = []

for epoch in range(3000):
    pred = model(x_train)
    loss = torch.mean((pred - y_train)**2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    if epoch % 500 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.3e}")

In [ ]:
x_test = torch.linspace(-torch.pi, torch.pi, 300, device=device).reshape(-1, 1)
with torch.no_grad():
    y_pred = model(x_test)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(x_test.cpu(), torch.sin(x_test).cpu(), label="exact")
plt.plot(x_test.cpu(), y_pred.cpu(), "--", label="MLP")
plt.legend()
plt.title("Function fitting")

plt.subplot(1, 2, 2)
plt.semilogy(loss_history)
plt.title("Loss history")
plt.tight_layout()
plt.show()

## 3. 小结

普通监督学习依赖成对数据 `(x, y)`。PINN 会把微分方程本身也变成监督信号：即使没有很多真实解数据，也能要求网络输出满足物理方程。下一节开始正式解 ODE。